# Standard KITTI 3D LiDAR training (Google Colab)

Notebook huấn luyện độc lập các biến thể loss ablation B0–B5. Chỉ sửa cell **Configuration**; trainer khóa mỗi run theo provenance bất biến trước khi resume.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import math
import os
import torch

# The loss-ablation branch contains the registered B0–B5 snapshots.
BRANCH = "feature/loss-function"
VARIANT = "B5"  # B0–B5; B5 is the default L4 smoke candidate
CONFIG_OVERRIDE = None  # e.g. "configs/kitti/my_proposal.json"
RUN_NAME = None  # None creates a stable, resume-friendly name
SEED = 42
PRECISION = "bf16"  # fp32, fp16, bf16 (BF16 requires a supported GPU)
PHYSICAL_BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
TARGET_BACKEND = "numba"  # parity-tested; use python for the eager reference
COMPILE_MODEL = False  # Opt in only after timing a complete warm run
COMPILE_MODEL_ARGUMENT = "--compile-model" if COMPILE_MODEL else ""
RUNTIME_PROFILE = f"{TARGET_BACKEND}_{'compile' if COMPILE_MODEL else 'eager'}"

EPOCHS = 100  # Matches the B-series configs; must remain unchanged when resuming
NUM_WORKERS = max(0, min(6, (os.cpu_count() or 1) - 1))
RUN_SMOKE_TEST = True
SMOKE_TRAIN_BATCHES = 2
SMOKE_VAL_BATCHES = 1
RUN_EVALUATION = True

REPOSITORY_URL = "https://github.com/danhyoyo/Lidar.git"
REPO_DIR = Path("/content/Lidar")
KITTI_TAR_ROOT = Path("/content/drive/MyDrive/KITTI_DATASET_ZIP")
RAW_KITTI_ROOT = Path("/content/KITTI_DATASET")
PROCESSED_DATASET_DIR = REPO_DIR / "data/kitti/processed"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/lidar_training_artifacts")

drive.mount("/content/drive")
%cd /content
!test -d Lidar || git clone "{REPOSITORY_URL}" Lidar
if _exit_code:
    raise RuntimeError("Không clone được repository.")
%cd /content/Lidar
!git fetch --prune origin "+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không fetch được branch: {BRANCH}")
!git checkout --detach "origin/{BRANCH}"
if _exit_code:
    raise RuntimeError(f"Không checkout được branch: {BRANCH}")
COMMIT = !git rev-parse HEAD
COMMIT = COMMIT[0]
print(f"Branch: {BRANCH}\nCommit: {COMMIT}")

if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Runtime > Change runtime type trước khi train.")
if PRECISION == "bf16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("GPU này không hỗ trợ BF16; đổi PRECISION thành fp16 hoặc fp32.")
print(f"PyTorch: {torch.__version__}; GPU: {torch.cuda.get_device_name(0)}")

## Dependencies and variant

`CONFIG_OVERRIDE` is the escape hatch for a new proposal; it must be a repository-relative JSON path.

In [ ]:
%cd /content/Lidar
%pip install -q "numba>=0.59" shapely onnx tensorboard tqdm

VARIANT_CONFIGS = {
    "B0": "configs/kitti/loss_ablation/b0_a4_legacy_uwag.json",
    "B1": "configs/kitti/loss_ablation/b1_l1_fixed.json",
    "B2": "configs/kitti/loss_ablation/b2_kfiou_fixed.json",
    "B3": "configs/kitti/loss_ablation/b3_probiou_fixed.json",
    "B4": "configs/kitti/loss_ablation/b4_kld_fixed.json",
    "B5": "configs/kitti/loss_ablation/b5_mgiou_fixed.json",
}
if CONFIG_OVERRIDE is None and VARIANT not in VARIANT_CONFIGS:
    raise ValueError(f"Unknown VARIANT={VARIANT!r}; use CONFIG_OVERRIDE for a new proposal.")
CONFIG_RELATIVE = CONFIG_OVERRIDE or VARIANT_CONFIGS[VARIANT]
CONFIG = (REPO_DIR / CONFIG_RELATIVE).resolve()
if REPO_DIR not in CONFIG.parents or not CONFIG.is_file():
    raise FileNotFoundError(f"Config unavailable on {BRANCH}: {CONFIG_RELATIVE}")
CONFIG_JSON = json.loads(CONFIG.read_text(encoding="utf-8"))
SELECTION_POLICY = CONFIG_JSON.get("train", {}).get("selection_policy", "best")
SELECTED_CHECKPOINT_FILENAME = {"best": "best.pt", "final_epoch": "final.pt"}.get(SELECTION_POLICY)
if SELECTED_CHECKPOINT_FILENAME is None:
    raise ValueError(f"Unsupported train.selection_policy: {SELECTION_POLICY!r}")
EFFECTIVE_BATCH_SIZE = PHYSICAL_BATCH_SIZE * ACCUMULATION_STEPS
RUN_NAME = RUN_NAME or f"{VARIANT.lower()}_seed{SEED}_eb{EFFECTIVE_BATCH_SIZE}_{RUNTIME_PROFILE}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Variant: {VARIANT}\nConfig: {CONFIG_RELATIVE}\nRun: {RUN_NAME}\nRuntime: {RUNTIME_PROFILE}")

## Prepare KITTI

The cell is idempotent: complete folders are reused; incomplete folders are re-extracted and validated.

In [ ]:
archives = {
    "velodyne": ("*.bin", 7481),
    "label_2": ("*.txt", 7481),
    "calib": ("*.txt", 7481),
}
RAW_KITTI_ROOT.mkdir(parents=True, exist_ok=True)

for folder, (pattern, expected_count) in archives.items():
    archive = KITTI_TAR_ROOT / f"{folder}.tar"
    extracted_dir = RAW_KITTI_ROOT / "training" / folder
    extracted_count = sum(1 for _ in extracted_dir.glob(pattern))

    if extracted_count != expected_count:
        if not archive.is_file():
            raise FileNotFoundError(f"Không tìm thấy archive: {archive}")
        archive_bytes = archive.stat().st_size
        !set -o pipefail; python3 -m tqdm --bytes --total {archive_bytes} --desc "Giải nén {folder}" < "{archive}" | tar --no-same-owner -xf - -C "{RAW_KITTI_ROOT}"
        if _exit_code:
            raise RuntimeError(f"Giải nén thất bại: {archive}")

    actual_count = sum(1 for _ in extracted_dir.glob(pattern))
    if actual_count != expected_count:
        raise RuntimeError(f"{folder}: {actual_count} file, cần {expected_count}")

pointcloud_dir = PROCESSED_DATASET_DIR / "pointcloud"
label_dir = PROCESSED_DATASET_DIR / "label"
dataset_ready = (
    sum(1 for _ in pointcloud_dir.glob("*.bin")) == 7481
    and sum(1 for _ in label_dir.glob("*.txt")) == 7481
    and (PROCESSED_DATASET_DIR / "train.txt").is_file()
    and (PROCESSED_DATASET_DIR / "val.txt").is_file()
)

if not dataset_ready:
    %cd /content/Lidar
    !python3 tools/kitti_training_pipeline/prepare_kitti.py \
      --kitti-root "{RAW_KITTI_ROOT}" \
      --output-root "{PROCESSED_DATASET_DIR}" \
      --config-output "{REPO_DIR / 'data/kitti/generated_kitti.json'}" \
      --train-ids "{REPO_DIR / 'splits/kitti/train.txt'}" \
      --val-ids "{REPO_DIR / 'splits/kitti/val.txt'}" \
      --pointcloud-mode symlink \
      --overwrite
    if _exit_code:
        raise RuntimeError("prepare_kitti.py thất bại")

pointcloud_count = sum(1 for _ in pointcloud_dir.glob("*.bin"))
label_count = sum(1 for _ in label_dir.glob("*.txt"))
assert pointcloud_count == label_count == 7481
assert (PROCESSED_DATASET_DIR / "train.txt").is_file()
assert (PROCESSED_DATASET_DIR / "val.txt").is_file()

print(f"Raw KITTI: {RAW_KITTI_ROOT}")
print(f"Processed: {PROCESSED_DATASET_DIR}")
print(f"Frames: {pointcloud_count}")


## Verify the checked-out loss ablation

Run this before the smoke/full training cells, so the checked-out B-series branch proves its loss and notebook contracts.

In [ ]:
%cd /content/Lidar
!MPLCONFIGDIR=/tmp/lidar-mpl PYTHONPATH=. python3 -m unittest discover -s tests -p 'test_loss_ablation.py' && MPLCONFIGDIR=/tmp/lidar-mpl PYTHONPATH=. python3 -m unittest discover -s tests -p 'test_standard_training_notebook.py'
if _exit_code:
    raise RuntimeError("B-series loss/notebook tests failed; do not train this checkout.")

## Smoke test

This uses a separate immutable trainer run. Rerunning the notebook resumes it only after provenance and checkpoint sidecar checks.

In [ ]:
if RUN_SMOKE_TEST:
    SMOKE_RUN_NAME = f"{RUN_NAME}_smoke"
    SMOKE_RUN_DIR = ARTIFACT_ROOT / SMOKE_RUN_NAME
    SMOKE_CHECKPOINT_DIR = SMOKE_RUN_DIR / "checkpoints"
    SMOKE_RESOLVED_CONFIG_PATH = SMOKE_RUN_DIR / "config.resolved.json"
    SMOKE_MANIFEST_PATH = SMOKE_RUN_DIR / "run.manifest.json"
    SMOKE_METRICS = SMOKE_RUN_DIR / "metrics.jsonl"
    smoke_last_checkpoint = SMOKE_CHECKPOINT_DIR / "last.pt"
    if SMOKE_RUN_DIR.exists():
        if not SMOKE_RESOLVED_CONFIG_PATH.is_file() or not SMOKE_MANIFEST_PATH.is_file():
            raise RuntimeError("Existing smoke run has no trainer provenance; choose a new RUN_NAME.")
        if not smoke_last_checkpoint.is_file():
            raise RuntimeError("Existing smoke run has no checkpoints/last.pt; choose a new RUN_NAME.")
        smoke_checkpoint_sidecar = smoke_last_checkpoint.with_suffix(smoke_last_checkpoint.suffix + ".sha256.json")
        if not smoke_checkpoint_sidecar.is_file():
            raise RuntimeError(f"Missing smoke checkpoint SHA256 sidecar: {smoke_checkpoint_sidecar}")
        SMOKE_RESUME_ARGUMENT = f'--resume "{smoke_last_checkpoint}"'
        print(f"Resuming smoke trainer run from verified candidate: {smoke_last_checkpoint}")
    else:
        SMOKE_RESUME_ARGUMENT = ""
        print(f"Starting fresh smoke trainer run: {SMOKE_RUN_DIR}")
    !python3 tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{SMOKE_RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs 1 --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --max-train-batches {SMOKE_TRAIN_BATCHES} --max-val-batches {SMOKE_VAL_BATCHES} --num-workers 0 --target-backend "{TARGET_BACKEND}" {COMPILE_MODEL_ARGUMENT} {SMOKE_RESUME_ARGUMENT}
    if _exit_code:
        raise RuntimeError(f"Smoke test thất bại cho runtime {RUNTIME_PROFILE}.")
    if not smoke_last_checkpoint.is_file() or not SMOKE_METRICS.is_file():
        raise RuntimeError("Smoke test không tạo đủ checkpoint/metrics.")
    smoke_rows = [line for line in SMOKE_METRICS.read_text(encoding="utf-8").splitlines() if line]
    smoke_row = json.loads(smoke_rows[-1]) if smoke_rows else None
    if not smoke_row or smoke_row["epoch"] != 1:
        raise RuntimeError("Smoke metrics thiếu epoch 1.")
    if smoke_row["epoch_successful_updates"] < 1:
        raise RuntimeError("Smoke test không thực hiện optimizer update.")
    if not math.isfinite(smoke_row["train_objective"]):
        raise RuntimeError("Smoke train objective không hữu hạn.")
    if not math.isfinite(smoke_row["validation"]["loss"]):
        raise RuntimeError("Smoke validation loss không hữu hạn.")
    print(f"Smoke PASS [{RUNTIME_PROFILE}]: {smoke_last_checkpoint}")

## Full training / safe resume

The trainer writes `checkpoints/last.pt`, appends one row to `metrics.csv`, and updates TensorBoard in `tensorboard/` after every completed epoch. Training stays attached to this cell; if interrupted, rerun it to resume from the latest completed epoch.

In [ ]:
RUN_DIR = ARTIFACT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
TRAIN_LOG = ARTIFACT_ROOT / f"{RUN_NAME}.train.log"
RESOLVED_CONFIG_PATH = RUN_DIR / "config.resolved.json"
MANIFEST_PATH = RUN_DIR / "run.manifest.json"
last_checkpoint = CHECKPOINT_DIR / "last.pt"
if RUN_DIR.exists():
    if not RESOLVED_CONFIG_PATH.is_file() or not MANIFEST_PATH.is_file():
        raise RuntimeError("Existing run has no trainer provenance; choose a new RUN_NAME.")
    if not last_checkpoint.is_file():
        raise RuntimeError("Existing run has no checkpoints/last.pt; choose a new RUN_NAME.")
    checkpoint_sidecar = last_checkpoint.with_suffix(last_checkpoint.suffix + ".sha256.json")
    if not checkpoint_sidecar.is_file():
        raise RuntimeError(f"Missing checkpoint SHA256 sidecar: {checkpoint_sidecar}")
    RESUME_ARGUMENT = f'--resume "{last_checkpoint}"'
    print(f"Resuming trainer run from verified candidate: {last_checkpoint}")
else:
    RESUME_ARGUMENT = ""
    print(f"Starting fresh trainer run: {RUN_DIR}")
!set -o pipefail; python3 -u tools/kitti_training_pipeline/train.py --config "{CONFIG}" --detector-root detector --output-root "{ARTIFACT_ROOT}" --run-name "{RUN_NAME}" --device cuda --precision "{PRECISION}" --seed {SEED} --epochs {EPOCHS} --physical-batch-size {PHYSICAL_BATCH_SIZE} --accumulation-steps {ACCUMULATION_STEPS} --num-workers {NUM_WORKERS} --target-backend "{TARGET_BACKEND}" {COMPILE_MODEL_ARGUMENT} {RESUME_ARGUMENT} 2>&1 | tee -a "{TRAIN_LOG}"
if _exit_code:
    raise RuntimeError(f"Training failed with exit code {_exit_code}")

## Select checkpoint and evaluate

The checkpoint path is derived from the selected config policy; B0–B5 use `final_epoch`, hence `selected/final.pt`.

In [ ]:
CHECKPOINT = RUN_DIR / "selected" / SELECTED_CHECKPOINT_FILENAME
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
CHECKPOINT_SIDECAR = CHECKPOINT.with_suffix(CHECKPOINT.suffix + ".sha256.json")
if not CHECKPOINT_SIDECAR.is_file():
    raise FileNotFoundError(CHECKPOINT_SIDECAR)

if RUN_EVALUATION:
    split = REPO_DIR / "splits/kitti/val.txt"
    output = RUN_DIR / "evaluation_validation.json"
    !python3 tools/kitti_training_pipeline/evaluate_kitti_bev.py --name "{RUN_NAME}_validation" --backend pytorch --model "{CHECKPOINT}" --config "{CONFIG}" --detector-root detector --kitti-root "{RAW_KITTI_ROOT}" --split "{split}" --output "{output}" --device cuda --warmup-frames 10
    if _exit_code:
        raise RuntimeError("Đánh giá validation thất bại.")
print(f"Checkpoint: {CHECKPOINT}")